# 📚 LeetCode 二分查找 & SQL窗口函数 复习笔记

> **题目**：LC 704 · LC 35 · LC 185  
> **题型**：标准二分查找 · 二分查找（求左边界/插入点）· SQL（窗口函数 DENSE_RANK + PARTITION BY）  
> **用途**：面试前快速回忆题型、触发条件、模板

---

## 目录

| # | 题号 | 题名 | 题型 | 难度 |
|---|------|------|------|------|
| 1 | 704 | 二分查找 | 标准二分查找（三分支）| 🟢 简单 |
| 2 | 35  | 搜索插入位置 | 二分查找（求左边界/lower bound）| 🟢 简单 |
| 3 | 185 | 部门工资前三高的员工 | SQL（窗口函数 DENSE_RANK + PARTITION BY + JOIN）| 🔴 困难 |
| 4 | —   | 模式对照总结 | — | — |

In [ ]:
from typing import List
import pandas as pd

print("环境就绪 ✅")

---
<a id='lc704'></a>
## 1. LC 704 — 二分查找（标准三分支二分）

| 字段 | 内容 |
|------|------|
| **题号** | 704 |
| **难度** | 🟢 简单 |
| **题型** | 标准二分查找（三分支判断）|
| **时间复杂度** | **O(log n)**（题目强制要求） |
| **空间复杂度** | O(1) |
| **数据范围** | 1 ≤ n ≤ 10⁴，−9999 ≤ nums[i], target ≤ 9999，**nums 升序且元素不重复** |

### 题目核心
在升序数组中查找 `target`，存在则返回下标，不存在返回 `-1`。

### ⚡ 触发条件（何时用二分查找？）

| # | 信号 | 说明 |
|---|------|------|
| 1 | 数组**已排序**（或存在某种单调性）| 二分的根基：能通过比较中点值，排除一半区间 |
| 2 | 要求 **O(log n)** 时间复杂度 | 排除 O(n) 线性扫描，强烈提示二分 |
| 3 | "查找某个值是否存在" / "查找某个边界" | 标准二分 vs 边界二分（见 LC 35 对比）|

---

### 🧠 算法（闭区间写法 `[lk, rk]`）

```
lk, rk = 0, n - 1

while lk <= rk:                  # 注意：<=，因为是闭区间，lk==rk时区间仍有1个元素需检查
    mid = (lk + rk) // 2

    if target > nums[mid]:
        lk = mid + 1              # target 在右半区，收缩左边界
    elif target < nums[mid]:
        rk = mid - 1              # target 在左半区，收缩右边界
    else:                          # target == nums[mid]
        return mid                # 找到，直接返回

return -1                          # 循环结束仍未找到
```

---

### 📐 区间写法的两种范式（重要！）

| | 闭区间 `[lk, rk]`（本题写法）| 左闭右开 `[lk, rk)` |
|--|--|--|
| 初始化 | `lk, rk = 0, n-1` | `lk, rk = 0, n` |
| 循环条件 | `while lk <= rk` | `while lk < rk` |
| 收缩右边界 | `rk = mid - 1`（mid 已排除）| `rk = mid`（mid 可能仍是答案）|
| 收缩左边界 | `lk = mid + 1` | `lk = mid + 1` |

> ⚠️ **最容易出错的地方**：`<=` vs `<`、`mid-1` vs `mid`。  
> 两种写法**不能混用**——闭区间用了 `rk = mid` 会导致死循环（mid 已经判断过，不应再留在区间内）。  
> **建议**：固定使用一种写法（本笔记统一用**闭区间**），减少混淆。

---

### 🔍 mid 计算的细节

```python
mid = (lk + rk) // 2        # 可能在 lk+rk 很大时溢出（Python 无此问题，但 Java/C++ 要注意）
mid = lk + (rk - lk) // 2   # 防溢出写法（面试中提及会加分）
```

In [ ]:
# ===== 通用模板：标准二分查找（闭区间，查找是否存在）=====
# 触发：有序数组 + 查找某值是否存在 + O(log n)
# 复杂度：O(log n) time | O(1) space

def binary_search_template(nums: List[int], target: int) -> int:
    lk, rk = 0, len(nums) - 1

    while lk <= rk:
        mid = (lk + rk) // 2

        if target > nums[mid]:
            lk = mid + 1      # target 偏大 → 收缩左边界
        elif target < nums[mid]:
            rk = mid - 1      # target 偏小 → 收缩右边界
        else:
            return mid        # 命中

    return -1   # 区间为空，未找到

print("✅ 标准二分查找模板已加载")

In [ ]:
# ===== LC 704 完整解法 =====
# O(log n) time | O(1) space
# 闭区间 [lk, rk]，三分支判断

class Solution_704:
    def search(self, nums: List[int], target: int) -> int:
        n = len(nums)
        lk, rk = 0, n - 1

        while lk <= rk:
            mid = (lk + rk) // 2

            if target > nums[mid]:
                lk = mid + 1
            if target < nums[mid]:
                rk = mid - 1
            if target == nums[mid]:
                return mid

        return -1

print("✅ LC 704 解法已加载")

In [ ]:
# ===== 测试用例 =====

sol704 = Solution_704()

tests704 = [
    ([-1,0,3,5,9,12],  9,  4),    # 示例1：命中
    ([-1,0,3,5,9,12],  2, -1),    # 示例2：不存在
    ([-1,0,3,5,9,12], -1,  0),    # 边界：最左元素
    ([-1,0,3,5,9,12], 12,  5),    # 边界：最右元素
    ([5],              5,  0),    # 单元素：命中
    ([5],              3, -1),    # 单元素：未命中
]

all_pass = True
for nums, target, expected in tests704:
    result = sol704.search(nums, target)
    ok = result == expected
    if not ok:
        all_pass = False
    flag = "✅" if ok else "❌"
    print(f"{flag}  nums={nums}, target={target}  →  {result}  (期望 {expected})")

print()
print("🎉 全部通过！" if all_pass else "⚠️ 有用例未通过")

---
<a id='lc35'></a>
## 2. LC 35 — 搜索插入位置（二分查找求左边界 / lower bound）

| 字段 | 内容 |
|------|------|
| **题号** | 35 |
| **难度** | 🟢 简单 |
| **题型** | 二分查找 — 求左边界（Lower Bound）|
| **时间复杂度** | O(log n) |
| **空间复杂度** | O(1) |
| **数据范围** | 1 ≤ n ≤ 10⁴，−10⁴ ≤ nums[i], target ≤ 10⁴，**nums 升序且不重复** |

### 题目核心
在升序数组中找 `target`；若存在返回其下标，**若不存在则返回它应被插入的位置**  
（即第一个 **≥ target** 的元素下标，也叫 "lower bound"）。

### ⚡ 触发条件

| 信号 | 说明 |
|------|------|
| 有序数组 + O(log n) | 二分查找的基本信号（同 LC 704）|
| **不只是"存在与否"，而是要确定一个边界位置** | 即使 target 不存在，也要返回有意义的位置 → **lower bound 二分** |
| 关键词："第一个 ≥/>"、"最后一个 ≤/<"、"插入位置" | 都是边界二分的变体 |

---

### 🧠 与 LC 704 的核心区别

| | LC 704（标准二分）| LC 35（lower bound）|
|--|--|--|
| 找到 target 时 | `return mid`（立即返回）| `return mid`（立即返回，同样有效）|
| **未找到 target 时** | `return -1` | `return lk`（**lk 最终停在"第一个 ≥ target"的位置**）|
| 循环结束后 lk, rk 的关系 | 无意义（已 return）| **lk = rk + 1**，且 `lk` 就是答案 |

**为什么 `return lk` 是正确的插入位置？**

二分结束时，区间 `[lk, rk]` 为空（`lk = rk+1`），此时：
- `nums[0..rk]` 中所有元素 **< target**（都被 `lk = mid+1` 这条路径排除）
- `nums[lk..n-1]` 中所有元素 **≥ target**（都被 `rk = mid-1` 这条路径排除）

所以 `lk` 正是"第一个 ≥ target 的下标"——也就是 target 应插入的位置。

---

### 🧠 算法（闭区间写法，与 LC 704 几乎一致！）

```
lk, rk = 0, n - 1

while lk <= rk:
    mid = (lk + rk) // 2

    if target > nums[mid]:
        lk = mid + 1
    elif target == nums[mid]:
        return mid              # 命中，直接返回下标
    else:                        # target < nums[mid]
        rk = mid - 1

return lk                        # 未命中：lk 即插入位置
```

> 💡 **记忆技巧**：LC 704 和 LC 35 的二分主循环**代码几乎一样**，  
> 唯一区别是最后一行：`return -1` （只关心存在性）vs `return lk`（关心边界位置）。  
> 掌握了这一点，二分模板可以"一套代码应对两种题"。

---

### 🗺️ Lower Bound / Upper Bound 速查（拓展）

| 需求 | 二分查找目标 |
|------|-------------|
| 第一个 ≥ target 的下标（本题）| lower_bound |
| 第一个 > target 的下标 | upper_bound（把 `target == nums[mid]` 归入"偏小"分支即可）|
| 最后一个 ≤ target 的下标 | `upper_bound - 1` |
| 最后一个 < target 的下标 | `lower_bound - 1` |

> 经典应用：LC 34（在排序数组中查找元素的第一个和最后一个位置）  
> = 分别求 `target` 的 lower_bound 和 `target+1` 的 lower_bound（再减1）。

In [ ]:
# ===== 通用模板：二分查找 lower bound（第一个 >= target 的下标）=====
# 触发：有序数组 + 需要确定边界位置（即使target不存在）+ O(log n)
# 复杂度：O(log n) time | O(1) space

def lower_bound_template(nums: List[int], target: int) -> int:
    lk, rk = 0, len(nums) - 1

    while lk <= rk:
        mid = (lk + rk) // 2

        if target > nums[mid]:
            lk = mid + 1       # mid 及左侧均 < target，排除
        elif target == nums[mid]:
            return mid          # 命中
        else:                   # target < nums[mid]
            rk = mid - 1       # mid 及右侧均 >= target，但mid本身可能是答案，先记录后排除

    return lk   # 循环结束，lk = 第一个 >= target 的下标（插入位置）

print("✅ Lower Bound 模板已加载")

In [ ]:
# ===== LC 35 完整解法 =====
# O(log n) time | O(1) space
# 与 LC 704 几乎相同，唯一区别：最后 return lk 而非 -1

class Solution_35:
    def searchInsert(self, nums: List[int], target: int) -> int:
        n = len(nums)
        lk, rk = 0, n - 1

        while lk <= rk:
            mid = (lk + rk) // 2

            if target > nums[mid]:
                lk = mid + 1
            elif target == nums[mid]:
                return mid
            else:
                rk = mid - 1

        return lk   # 未命中：lk 即为插入位置

print("✅ LC 35 解法已加载")

In [ ]:
# ===== 测试用例 =====

sol35 = Solution_35()

tests35 = [
    ([1,3,5,6], 5, 2),   # 示例1：命中，在中间
    ([1,3,5,6], 2, 1),   # 示例2：未命中，插入到下标1
    ([1,3,5,6], 7, 4),   # 示例3：未命中，插入到末尾
    ([1,3,5,6], 0, 0),   # 边界：插入到开头
    ([],        5, 0),   # 空数组：插入到唯一位置0
    ([1],       1, 0),   # 单元素：命中
    ([1],       2, 1),   # 单元素：插入到末尾
]

all_pass = True
for nums, target, expected in tests35:
    result = sol35.searchInsert(nums, target)
    ok = result == expected
    if not ok:
        all_pass = False
    flag = "✅" if ok else "❌"
    print(f"{flag}  nums={nums}, target={target}  →  {result}  (期望 {expected})")

print()
print("🎉 全部通过！" if all_pass else "⚠️ 有用例未通过")

---
<a id='lc185'></a>
## 3. LC 185 — 部门工资前三高的员工（SQL窗口函数：DENSE_RANK + PARTITION BY + JOIN）

| 字段 | 内容 |
|------|------|
| **题号** | 185 |
| **难度** | 🔴 困难 |
| **题型** | SQL — 窗口函数分组排名（DENSE_RANK + PARTITION BY）+ 子查询 + JOIN |
| **时间复杂度** | O(n log n)（窗口函数内部按分区排序）|
| **空间复杂度** | O(n) |

### 题目核心
查询每个部门**工资前三高（按不同薪资计，允许并列）**的所有员工信息：  
姓名、薪资、所属部门名称。

### ⚡ 触发条件

| 信号 | 操作 |
|------|------|
| "**每个分组**的前 N 名 / 第 N 高" | 窗口函数 + `PARTITION BY 分组列` |
| 排名需考虑**并列**（如本题"前三高"按不同薪资算，并列不占额外名额）| `DENSE_RANK()`（而非 `ROW_NUMBER()` 或 `RANK()`）|
| 需要**跨表**字段（员工名 + 部门名）| `JOIN`（在窗口函数子查询**之后**做，避免 JOIN 影响分区计算）|

---

### 🧠 三种排名窗口函数对比（高频面试题！）

假设某部门薪资为 `[90000, 85000, 85000, 70000]`：

| 函数 | 90000 | 85000 | 85000 | 70000 | 特点 |
|------|-------|-------|-------|-------|------|
| `ROW_NUMBER()` | 1 | 2 | 3 | 4 | 并列也分配不同序号（无重复）|
| `RANK()` | 1 | 2 | 2 | **4** | 并列同名次，**跳过**后续名次（留空位）|
| `DENSE_RANK()` | 1 | 2 | 2 | **3** | 并列同名次，**不跳**后续名次（连续）|

**本题为何用 `DENSE_RANK`**：题目定义"前三高"是指**最多3个不同薪资值**，  
70000 应被视为"第3高的薪资"（即使前面有2人并列第2），因此用 `DENSE_RANK` 让其 rk=3。  
若用 `RANK`，70000 的 rk 会是 4，被错误地排除在"前三"之外。

---

### 🧠 算法结构

```sql
SELECT e.name AS Employee, e.salary AS Salary, d.name AS Department
FROM (
    SELECT departmentId, name, salary,
           DENSE_RANK() OVER (PARTITION BY departmentId ORDER BY salary DESC) AS rk
    FROM Employee
) e
JOIN Department d ON d.id = e.departmentId
WHERE e.rk <= 3;
```

**执行顺序拆解**：

1. **内层子查询**：对 `Employee` 按 `departmentId` 分区（`PARTITION BY`），  
   每个分区内部按 `salary DESC` 计算 `DENSE_RANK`，得到带 `rk` 列的中间表。
2. **JOIN**：把中间表与 `Department` 表连接，补充部门名称。
3. **外层 WHERE**：过滤 `rk <= 3`，即每个部门只保留前三档薪资的员工。

---

### 🔑 PARTITION BY vs GROUP BY（核心区别）

| | `GROUP BY` | `PARTITION BY`（窗口函数）|
|--|--|--|
| 输出行数 | 每组**一行**（聚合后）| **保留原始行数**，每行多一列"组内信息" |
| 能否同时看到明细 + 排名 | ❌ 不能（聚合丢失明细）| ✅ 能（每行都有 rk，明细仍保留）|
| 适用场景 | "每组的总数/平均值"等汇总 | "每组的前N名"等既要排名又要明细 |

> **本题的关键**：要返回**员工明细**（姓名、薪资），同时按部门分组排名 → 必须用 `PARTITION BY`，  
> 若用 `GROUP BY departmentId` 会直接丢失单个员工的信息。

---

### ⚠️ 易错点：WHERE 不能直接用窗口函数结果

```sql
-- ❌ 错误：WHERE 中不能直接引用同一层 SELECT 定义的窗口函数别名
SELECT departmentId, name, salary,
       DENSE_RANK() OVER (...) AS rk
FROM Employee
WHERE rk <= 3;   -- 报错！rk 在 WHERE 阶段还未计算完成

-- ✅ 正确：必须包一层子查询，在外层 WHERE 中引用
SELECT * FROM (
    SELECT departmentId, name, salary,
           DENSE_RANK() OVER (...) AS rk
    FROM Employee
) t
WHERE rk <= 3;
```

**原因**：SQL 执行顺序为 `FROM → WHERE → ... → SELECT`（窗口函数在 SELECT/ORDER BY 阶段计算），  
`WHERE` 执行时窗口函数结果尚不存在，因此必须借助子查询"物化"后再过滤——这与 LC 596 中  
"`HAVING` 不能直接用于聚合前的过滤"是同一类"执行顺序"问题。

In [ ]:
# ===== LC 185 — Pandas 模拟解法 =====
# 对应 SQL: DENSE_RANK() OVER (PARTITION BY departmentId ORDER BY salary DESC) + JOIN + WHERE rk<=3

import pandas as pd

def department_top_three_salaries(employee: pd.DataFrame, department: pd.DataFrame) -> pd.DataFrame:
    df = employee.copy()

    # 1) PARTITION BY departmentId, ORDER BY salary DESC → DENSE_RANK
    #    rank(method='dense') 对应 SQL 的 DENSE_RANK()
    df['rk'] = (
        df.groupby('departmentId')['salary']
          .rank(method='dense', ascending=False)
    )

    # 2) WHERE rk <= 3
    df = df[df['rk'] <= 3]

    # 3) JOIN Department
    result = df.merge(department, left_on='departmentId', right_on='id')

    # 4) 整理输出列名
    result = result.rename(columns={
        'name_x': 'Employee',
        'salary': 'Salary',
        'name_y': 'Department'
    })[['Department', 'Employee', 'Salary']]

    return result.reset_index(drop=True)

print("✅ Pandas 解法已加载")

In [ ]:
# ===== 测试用例 =====

# 构造 Employee 表（来自题目示例 + 截图中的 IT/Sales 数据）
employee = pd.DataFrame({
    'id':           [1,     2,     3,      4,     5,     6],
    'name':         ['Max', 'Joe', 'Randy','Will','Henry','Sam'],
    'salary':       [90000, 85000, 85000,  70000, 80000, 60000],
    'departmentId': [1,     1,     1,      1,     2,     2],
})

department = pd.DataFrame({
    'id':   [1,    2],
    'name': ['IT', 'Sales'],
})

print("输入 Employee：")
print(employee.to_string(index=False))
print()
print("输入 Department：")
print(department.to_string(index=False))
print()

result = department_top_three_salaries(employee, department)
print("查询结果（每部门前三高薪资的员工，DENSE_RANK含义）：")
print(result.to_string(index=False))
print()

# 验证：
# IT 部门: 90000(rk=1,Max), 85000(rk=2,Joe&Randy并列), 70000(rk=3,Will) → 4人全部入选
# Sales 部门: 80000(rk=1,Henry), 60000(rk=2,Sam) → 不足3档，全部入选
expected_employees = {'Max', 'Joe', 'Randy', 'Will', 'Henry', 'Sam'}
assert set(result['Employee']) == expected_employees, "应包含所有6名员工"
assert len(result) == 6
print("✅ 验证通过：")
print("  - IT 部门 70000(Will) 因 DENSE_RANK=3 被正确保留（若用RANK则会是4，被排除）")
print("  - Sales 部门仅2档薪资，全部保留（不足3档不影响结果）")

---
<a id='summary'></a>
## 4. 模式对照总结

### 三题横向对比

| | LC 704 | LC 35 | LC 185 |
|--|--|--|--|
| **核心题型** | 标准二分查找 | 二分查找（lower bound）| SQL窗口函数（DENSE_RANK+PARTITION BY）|
| **时间** | O(log n) | O(log n) | O(n log n) |
| **空间** | O(1) | O(1) | O(n) |
| **驱动逻辑** | 三分支判断，命中即返回 | 同三分支，但未命中返回 lk（左边界）| 分区内排名，物化后过滤 |
| **核心决策** | `>`→左移，`<`→右移，`==`→返回 | 与LC704相同主体，仅末尾 `return lk` 不同 | DENSE_RANK 处理并列 + 子查询绕过WHERE限制 |
| **易错点** | 闭区间 `<=` 与 `rk=mid-1` 配套，不可混用左闭右开写法 | 误以为找不到就返回-1，忘记"插入位置"语义 | WHERE 不能直接用窗口函数别名；RANK vs DENSE_RANK 选错导致并列处理错误 |

---

### 📐 二分查找统一模板（闭区间）

```python
# ── 标准二分（LC 704：只关心存在性）──────────────────
lk, rk = 0, n - 1
while lk <= rk:
    mid = (lk + rk) // 2
    if target > nums[mid]:   lk = mid + 1
    elif target < nums[mid]: rk = mid - 1
    else: return mid
return -1

# ── Lower Bound 二分（LC 35：关心边界位置）─────────────
lk, rk = 0, n - 1
while lk <= rk:
    mid = (lk + rk) // 2
    if target > nums[mid]:    lk = mid + 1
    elif target == nums[mid]: return mid
    else:                      rk = mid - 1
return lk    # 第一个 >= target 的下标
```

**两者唯一区别**：未命中时 `return -1`（只想知道在不在）  vs  `return lk`（想知道该插在哪）。

---

### 📐 SQL 窗口函数排名模板

```sql
SELECT <字段列表>
FROM (
    SELECT *,
           DENSE_RANK() OVER (PARTITION BY <分组列> ORDER BY <排序列> DESC) AS rk
    FROM <表>
) t
[JOIN <其他表> ON ...]
WHERE rk <= N;
```

| 场景 | 选用函数 |
|------|---------|
| 严格排名，不允许重复序号 | `ROW_NUMBER()` |
| 并列同名次，后续名次跳号 | `RANK()` |
| 并列同名次，后续名次不跳号（本题）| `DENSE_RANK()` |

---

### 🗺️ 题型识别速查

```
有序数组 + 查找值是否存在 + O(log n)            →  标准二分查找       (LC 704)
有序数组 + 找"第一个>=/最后一个<="边界位置       →  二分查找(lower/upper bound) (LC 35, LC 34)
"每组前N名"/"每组第N高" + 需保留明细行          →  窗口函数+PARTITION BY (LC 185, LC 184)
排名需处理"并列"情况                            →  DENSE_RANK / RANK 选择
```

---
*复习笔记 · 2026-06 · By Claude*